In [1]:
# COVID-19 Analysis and Forecasting Project
# Works with covid_19_clean_complete.csv dataset

import os
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# ---------------------------
# Load data
# ---------------------------

OUT_DIR = "covid_analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PATH = "covid_19_clean_complete.csv"  # Update if your file is in another location

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df.columns = [c.strip().replace(" ", "_").replace("/", "_") for c in df.columns]

# Rename if needed
rename_map = {}
if "Country_Region" in df.columns:
    rename_map["Country_Region"] = "Country"
if "Province_State" in df.columns:
    rename_map["Province_State"] = "Province"
if rename_map:
    df = df.rename(columns=rename_map)

# Ensure numeric columns
for c in ["Confirmed","Deaths","Recovered","Active"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

print("Dataset loaded.")
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

# ---------------------------
# Aggregations / EDA
# ---------------------------
global_daily = df.groupby("Date")[["Confirmed","Deaths","Recovered","Active"]].sum().reset_index()
latest_date = df["Date"].max()
latest_snapshot = df[df["Date"]==latest_date].groupby("Country")[["Confirmed","Deaths","Recovered","Active"]].sum().reset_index()
latest_snapshot = latest_snapshot.sort_values("Confirmed", ascending=False).reset_index(drop=True)

# Save top-10
top10 = latest_snapshot.head(10)
top10.to_csv(os.path.join(OUT_DIR, "top10_countries_latest.csv"), index=False)
print("\nTop 10 countries by confirmed (latest date):")
print(top10)

# ---------------------------
# Matplotlib plots
# ---------------------------
# Global trends
plt.figure(figsize=(12,6))
plt.plot(global_daily['Date'], global_daily['Confirmed'], label='Confirmed')
plt.plot(global_daily['Date'], global_daily['Deaths'], label='Deaths')
plt.plot(global_daily['Date'], global_daily['Recovered'], label='Recovered')
plt.xlabel("Date")
plt.ylabel("Count")
plt.title("Global COVID-19 Trends")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "global_trends_matplotlib.png"))
plt.close()

# Active cases
plt.figure(figsize=(12,5))
plt.plot(global_daily['Date'], global_daily['Active'])
plt.xlabel("Date")
plt.ylabel("Active cases")
plt.title("Global Active Cases")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "global_active_matplotlib.png"))
plt.close()

# India vs Rest-of-World
if "India" in df["Country"].unique():
    india_daily = df[df["Country"]=="India"].groupby("Date")[["Confirmed","Deaths","Recovered","Active"]].sum().reset_index()
    merged = global_daily[["Date","Confirmed"]].merge(
        india_daily[["Date","Confirmed"]].rename(columns={"Confirmed":"India_Confirmed"}),
        on="Date", how="left"
    ).fillna(0)
    merged["RestWorld_Confirmed"] = merged["Confirmed"] - merged["India_Confirmed"]

    plt.figure(figsize=(12,6))
    plt.plot(merged["Date"], merged["India_Confirmed"], label="India Confirmed")
    plt.plot(merged["Date"], merged["RestWorld_Confirmed"], label="Rest of World Confirmed")
    plt.xlabel("Date"); plt.ylabel("Confirmed cases")
    plt.title("India vs Rest of World - Confirmed")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "india_vs_restworld_confirmed.png"))
    plt.close()

# ---------------------------
# Plotly interactive charts
# ---------------------------
# Choropleth
fig_map = px.choropleth(
    latest_snapshot, locations="Country", locationmode="country names",
    color="Confirmed", hover_name="Country",
    title=f"Confirmed cases by country (as of {latest_date.date()})"
)
fig_map.write_html(os.path.join(OUT_DIR, "confirmed_choropleth.html"))

# Global time series
fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(x=global_daily['Date'], y=global_daily['Confirmed'], mode='lines', name='Confirmed'))
fig_ts.add_trace(go.Scatter(x=global_daily['Date'], y=global_daily['Deaths'], mode='lines', name='Deaths'))
fig_ts.add_trace(go.Scatter(x=global_daily['Date'], y=global_daily['Recovered'], mode='lines', name='Recovered'))
fig_ts.update_layout(title="Global COVID-19 Time Series", xaxis_title="Date", yaxis_title="Count")
fig_ts.write_html(os.path.join(OUT_DIR, "global_time_series.html"))

# ---------------------------
# Forecasting (Prophet if available, else fallback)
# ---------------------------
def try_get_prophet():
    try:
        from prophet import Prophet
        return Prophet
    except:
        try:
            from fbprophet import Prophet
            return Prophet
        except:
            return None

ProphetClass = try_get_prophet()

def forecast_with_prophet(series, periods=7):
    prophet_df = series.reset_index().rename(columns={"Date":"ds", series.name:"y"})
    model = ProphetClass(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True)
    model.fit(prophet_df)
    future = model.make_future_dataframe(periods=periods)
    fcst = model.predict(future)
    return fcst[['ds','yhat','yhat_lower','yhat_upper']]

def simple_forecast(series, periods=7):
    s = series.sort_index()
    daily_new = s.diff().fillna(0)
    mean_inc = daily_new.tail(7).mean()
    last_val = s.iloc[-1]
    preds = []
    for i in range(1, periods+1):
        last_val += mean_inc
        preds.append({"ds": s.index[-1] + timedelta(days=i), "yhat": last_val})
    return pd.DataFrame(preds)

# Global forecast
global_series = global_daily.set_index("Date")["Confirmed"].rename("Confirmed")
if ProphetClass:
    try:
        global_forecast = forecast_with_prophet(global_series, periods=7)
        method_global = "Prophet"
    except Exception as e:
        global_forecast = simple_forecast(global_series, 7)
        method_global = "Fallback (7-day avg)"
else:
    global_forecast = simple_forecast(global_series, 7)
    method_global = "Fallback (7-day avg)"

global_forecast.to_csv(os.path.join(OUT_DIR, "global_confirmed_forecast.csv"), index=False)
print(f"\nGlobal forecast method: {method_global}")
print(global_forecast)

# India forecast
if "India" in df["Country"].unique():
    india_series = df[df["Country"]=="India"].groupby("Date")["Confirmed"].sum()
    if ProphetClass:
        try:
            india_forecast = forecast_with_prophet(india_series, 7)
            method_india = "Prophet"
        except:
            india_forecast = simple_forecast(india_series, 7)
            method_india = "Fallback (7-day avg)"
    else:
        india_forecast = simple_forecast(india_series, 7)
        method_india = "Fallback (7-day avg)"

    india_forecast.to_csv(os.path.join(OUT_DIR, "india_confirmed_forecast.csv"), index=False)
    print(f"\nIndia forecast method: {method_india}")
    print(india_forecast)

print("\nAll results saved in:", OUT_DIR)


Dataset loaded.
Date range: 2020-01-22 to 2020-07-27

Top 10 countries by confirmed (latest date):
          Country  Confirmed  Deaths  Recovered   Active
0              US    4290259  148011    1325804  2816444
1          Brazil    2442375   87618    1846641   508116
2           India    1480073   33408     951166   495499
3          Russia     816680   13334     602249   201097
4    South Africa     452529    7067     274925   170537
5          Mexico     395489   44022     303810    47657
6            Peru     389717   18418     272547    98752
7           Chile     347923    9187     319954    18782
8  United Kingdom     301708   45844       1437   254427
9            Iran     293606   15912     255144    22550


DEBUG:cmdstanpy:input tempfile: /tmp/tmpm4mlv_kg/6w06jkl2.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpm4mlv_kg/unlwnb5x.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=11337', 'data', 'file=/tmp/tmpm4mlv_kg/6w06jkl2.json', 'init=/tmp/tmpm4mlv_kg/unlwnb5x.json', 'output', 'file=/tmp/tmpm4mlv_kg/prophet_modelh38wao_7/prophet_model-20250914070316.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
07:03:16 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
07:03:16 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
DEBUG:cmdstanpy:input tempfile: /tmp/tmpm4mlv_kg/2g3y_nzo.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpm4mlv_kg/cov3590f.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/


Global forecast method: Prophet
            ds          yhat    yhat_lower    yhat_upper
0   2020-01-22 -3.945555e+03 -1.494961e+04  6.928582e+03
1   2020-01-23  8.153520e+02 -1.038898e+04  1.127711e+04
2   2020-01-24  6.957575e+03 -3.920758e+03  1.773348e+04
3   2020-01-25  9.466119e+03 -2.182561e+03  2.077840e+04
4   2020-01-26  2.912911e+03 -7.764443e+03  1.374553e+04
..         ...           ...           ...           ...
190 2020-07-30  1.721597e+07  1.720449e+07  1.722675e+07
191 2020-07-31  1.743762e+07  1.742552e+07  1.744964e+07
192 2020-08-01  1.763797e+07  1.762371e+07  1.764989e+07
193 2020-08-02  1.780817e+07  1.779237e+07  1.782204e+07
194 2020-08-03  1.795223e+07  1.793617e+07  1.796696e+07

[195 rows x 4 columns]


07:03:17 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing



India forecast method: Prophet
            ds          yhat    yhat_lower    yhat_upper
0   2020-01-22 -3.863048e+02 -1.443740e+03  5.483181e+02
1   2020-01-23 -3.843175e+01 -1.016265e+03  9.424989e+02
2   2020-01-24  1.451172e+02 -7.805466e+02  1.152948e+03
3   2020-01-25  4.856316e+02 -5.597022e+02  1.444259e+03
4   2020-01-26  6.714745e+02 -3.214887e+02  1.638845e+03
..         ...           ...           ...           ...
190 2020-07-30  1.637422e+06  1.636355e+06  1.638442e+06
191 2020-07-31  1.688426e+06  1.687338e+06  1.689565e+06
192 2020-08-01  1.738587e+06  1.737358e+06  1.739797e+06
193 2020-08-02  1.787030e+06  1.785704e+06  1.788285e+06
194 2020-08-03  1.832457e+06  1.830951e+06  1.833970e+06

[195 rows x 4 columns]

All results saved in: covid_analysis_outputs
